In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace

from typing import TypedDict

from dotenv import load_dotenv

In [29]:
load_dotenv()

True

In [30]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct"
,  # Or another model of your choice
    task="text-generation",
    max_new_tokens=512
)
model = ChatHuggingFace(llm=llm)


In [32]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [33]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [34]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [35]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). This is called the "lunar distance" or "lunar mean distance." However, the Moon's orbit is not a perfect circle and its distance from Earth varies slightly due to the elliptical shape of its orbit.

At its closest point, called "perigee," the Moon is about 356,400 kilometers (221,500 miles) away from Earth. At its farthest point, called "apogee," the Moon is about 405,500 kilometers (252,000 miles) away from Earth.

It's worth noting that the Moon is slowly moving away from Earth at a rate of about 3.8 centimeters (1.5 inches) per year. This is because the Moon's orbit is gradually increasing in size due to the tidal interactions between the Earth and the Moon.


In [36]:
model.invoke('How far is moon from the earth?').content

'The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). This is called the "lunar distance" or "lunar mean distance." However, the Moon\'s orbit is not a perfect circle and its distance from Earth varies slightly due to the elliptical shape of its orbit.\n\nAt its closest point, called "perigee," the Moon is about 356,400 kilometers (221,500 miles) away from Earth. At its farthest point, called "apogee," the Moon is about 405,500 kilometers (252,000 miles) away from Earth.\n\nSo, to summarize:\n\n* Average distance: 384,400 kilometers (238,900 miles)\n* Closest point (perigee): 356,400 kilometers (221,500 miles)\n* Farthest point (apogee): 405,500 kilometers (252,000 miles)'